# 02b — Measured-inlet dataset D1 (System I, Phase 3)

**Why.** The production posterior (notebook 36) takes the *measured* feed and coolant inlet temperatures of every window as
inputs. `data/observations.npz` (notebook 02, **D0**) was generated at the nominal inlet and carries no inlet measurement, so it
cannot feed the production posterior. D0 stays as it is (the nominal-inlet dataset: naive model, Claim 1 in notebook 07).
This notebook writes **D1 = `data/observations_measured_inlet.npz`**.

**Design (fixed before running; `scripts/ws_phase3.py`).** Closed-loop scenarios Sc1–Sc5 and Sc7 × 50 windows (Sc0 and Sc6 are **not**
in D1: user decision 2026-11-09), the same θ per scenario as notebook 02, protocol P2 (300-min history), N3 sensors, Sc7's +2 K T-sensor
drift after the sensor layer. Per window: the **true** inlet [Ci, Ti, Tci] drawn from the nb35/nb36 training distribution
(Ti, Tci ~ U(295, 299) K independently; Ci = 0.97 (1 + N(0, 0.01²)) mol/L, unmeasured) and the **measured** (Ti, Tci) = true + ε N(0, 1)
calibration offset (ε = 0.1 K, fixed per window) + window-mean white noise N(0, 0.2²/120) K, exactly nb35's sensor model. New seed
base (3 700 000): D1 never reuses the windows of notebook 02 or 35.

**Checks (runbook §2f; any failure stops the notebook).** Shapes and finiteness; true Ti, Tci within [295, 299] K; Ci relative sd within 20%
of 0.01; measured − true sd per channel within 10% of √(0.1² + 0.0183²) = 0.1017 K; regeneration from the stored seeds is bit-identical;
Sc7's drift is exactly +2 K; the production (34-D) and naive (32-D) features are finite and the production features equal
`nb36.features_from_raw`. Saturation fractions per scenario are reported, not gated (with the inlet varying the β = 0.85 scenarios
saturate often, notebook 34).


In [1]:
import json, os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "system_I" and NB_DIR.parent.name == "notebooks", f"run from notebooks/system_I (cwd {NB_DIR})"
ROOT = NB_DIR.parent.parent
for p in (ROOT / "src", ROOT / "scripts"):
    sys.path.insert(0, str(p))
os.environ["CSTR_SENSOR_MODEL"] = "N3"
import importlib, system_I_config, ws_protocols, ws_eval, ws_inlet, nb35_measured_inlet, nb36_production_measured_inlet, ws_phase3
for _m in (system_I_config, ws_protocols, ws_eval, ws_inlet, nb35_measured_inlet, nb36_production_measured_inlet, ws_phase3):
    importlib.reload(_m)
P3, M = ws_phase3, nb35_measured_inlet
print("Sensor model:", system_I_config.describe(), "| protocol:", M.PROTOCOL)
print("D1 will be written to", P3.D1_PATH.relative_to(ROOT))


Sensor model: N3: sigma_C=0.01 mol/L, sigma_T=0.2 K, sigma_Tc=0.2 K, sigma_Qc=2.0 L/min | protocol: P2
D1 will be written to data/observations_measured_inlet.npz


## 1. Generate D1 and run the checks

In [2]:
t0 = time.time()
d = P3.generate_d1()
print(f"simulated {d['x'].shape} windows in {time.time() - t0:.1f}s")
n = len(d["x"]); sid = d["scenario_id"]
assert d["x"].shape == (300, 120, 4) and np.isfinite(d["x"]).all(), d["x"].shape
assert sorted(set(sid.tolist())) == [1, 2, 3, 4, 5, 7] and all((sid == s).sum() == 50 for s in (1, 2, 3, 4, 5, 7))
it, im = d["inlet_true"], d["inlet_measured"]
assert it[:, 1:].min() >= 295.0 and it[:, 1:].max() <= 299.0, (it[:, 1:].min(), it[:, 1:].max())
ci_rel = it[:, 0].std() / it[:, 0].mean()
assert abs(ci_rel / 0.01 - 1) < 0.20, ci_rel
dev = im - it[:, 1:]
want = float(np.sqrt(0.1 ** 2 + M.WHITE_SD ** 2))
assert np.all(np.abs(dev.std(axis=0) / want - 1) < 0.10), (dev.std(axis=0), want)
print(f"true Ti, Tci in [{it[:,1:].min():.2f}, {it[:,1:].max():.2f}] K | Ci relative sd {ci_rel:.4f} | measured - true sd per channel (Ti, Tci) = {np.round(dev.std(axis=0), 4)} (expected {want:.4f})")
print("corr(true Ti, true Tci) =", round(float(np.corrcoef(it[:, 1], it[:, 2])[0, 1]), 3), "(independent draws)")


simulated (300, 120, 4) windows in 2.8s
true Ti, Tci in [295.00, 298.99] K | Ci relative sd 0.0096 | measured - true sd per channel (Ti, Tci) = [0.1048 0.1053] (expected 0.1017)
corr(true Ti, true Tci) = -0.113 (independent draws)


In [3]:
# regeneration from the stored seeds is bit-identical; Sc7's drift is exactly +2 K
d2 = P3.generate_d1()
for k in ("x", "inlet_true", "inlet_measured", "theta", "scenario_id"):
    assert np.array_equal(d[k], d2[k]), k
import ws_inlet as wi
m7 = sid == 7
raw7 = np.asarray(wi.simulate(d["theta"][:, 2:], d["inlet_true"], M.PROTOCOL, int(d["seeds"][1]), raw=True, **M.NOISE))
drift = float((d["x"][m7][:, :, 1] - raw7[m7][:, :, 1]).mean())
assert abs(drift - 2.0) < 1e-3, drift
print(f"regeneration bit-identical: True | Sc7 drift added: {drift:.4f} K")
# features
f_prod = P3.production_features(d); f_naive = P3.naive_features(d)
assert f_prod.shape == (300, 34) and f_naive.shape == (300, 32) and np.isfinite(f_prod).all() and np.isfinite(f_naive).all()
ref = nb36_production_measured_inlet.features_from_raw(d["x"], d["inlet_measured"])
assert np.array_equal(f_prod, ref)
print("features: production", f_prod.shape, "naive", f_naive.shape, "| all finite, production equals nb36.features_from_raw")


regeneration bit-identical: True | Sc7 drift added: 2.0000 K
features: production (300, 34) naive (300, 32) | all finite, production equals nb36.features_from_raw


## 2. Saturation per scenario and the inlet statistics (reported, not gated)

In [4]:
from cstr_sbi.scenarios import SCENARIO_CONFIGS
rows = []
d0 = P3.load_d0()
for s in (1, 2, 3, 4, 5, 7):
    m = sid == s; q = d["x"][m][:, :, 3]; T = d["x"][m][:, :, 1]
    m0 = d0["scenario_id"] == s; q0 = d0["x"][m0][:, :, 3]
    rows.append(dict(scenario=f"Sc{s}", alpha=float(d["theta"][m][0, 2]), beta=float(d["theta"][m][0, 3]),
                     D1_frac_samples_Qc_ge395=float((q >= 395).mean()), D1_frac_windows_any_ge395=float((q >= 395).any(axis=1).mean()),
                     D0_frac_samples_Qc_ge395=float((q0 >= 395).mean()),
                     D1_Qc_mean=float(q.mean()), D0_Qc_mean=float(q0.mean()), D1_T_minus_Tsp=float(T.mean() - 312.5)))
sat = pd.DataFrame(rows); display(sat.round(3))
print("D0 = nominal inlet (observations.npz); D1 = inlet varying (+-2 K) with measured temperatures.")


,scenario,alpha,beta,D1_frac_samples_Qc_ge395,D1_frac_windows_any_ge395,D0_frac_samples_Qc_ge395,D1_Qc_mean,D0_Qc_mean,D1_T_minus_Tsp
0,Sc1,1.00,1.00,0.312,0.38,0.000,174.986,50.929,0.189
1,Sc2,1.00,0.85,0.431,0.46,0.000,221.593,130.651,0.353
2,Sc3,0.70,1.00,0.116,0.18,0.000,106.298,44.277,0.033
3,Sc4,0.85,0.85,0.304,0.34,0.000,168.021,115.716,0.177
4,Sc5,1.00,0.50,0.732,0.76,0.992,324.889,399.993,1.169
5,Sc7,1.00,0.85,0.378,0.42,0.000,195.644,139.355,2.313


D0 = nominal inlet (observations.npz); D1 = inlet varying (+-2 K) with measured temperatures.


## 3. Save D1

In [5]:
P3.D1_PATH.parent.mkdir(exist_ok=True)
np.savez(P3.D1_PATH, **d)
with np.load(P3.D1_PATH, allow_pickle=False) as f:
    keys = list(f.files); chk = {k: f[k] for k in f.files}
for k in ("x", "inlet_true", "inlet_measured", "theta"):
    assert np.array_equal(chk[k], d[k]), k
print("Saved", P3.D1_PATH.relative_to(ROOT), "| keys:", keys)
print("size %.2f MB" % (P3.D1_PATH.stat().st_size / 1e6))
sat.to_csv(ROOT / "results" / "02b_N3_saturation.csv", index=False)
print(f"notebook wall time {(time.time() - t0) / 60:.1f} min")


Saved data/observations_measured_inlet.npz | keys: ['theta', 'x', 'scenario_id', 'mode', 't', 'protocol', 'burn_min', 'controller_law', 'sensor_model', 'sensor_noise', 'inlet_true', 'inlet_measured', 'inlet_model', 'features', 'seeds']
size 0.61 MB
notebook wall time 0.1 min


## 4. Interpretation (written 2026-11-09)

**Result.** D1 (`data/observations_measured_inlet.npz`, 0.61 MB) is generated, passes every check and regenerates bit-identically from its seeds: 300 closed-loop windows (Sc1–Sc5, Sc7 × 50, protocol P2, N3 sensors), each with its **true** inlet [Ci, Ti, Tci] and its **measured** (Ti, Tci) (ε = 0.1 K offset plus window-mean noise), and the 34-D production and 32-D naive feature matrices are finite. The measured temperatures differ from the true ones by a standard deviation of 0.105 K for both channels (design value 0.102 K), the true Ti and Tci cover [295.00, 298.99] K, and the Ci relative spread is 0.0096 (design 0.01). Sc0 and Sc6 are not in D1 (user decision); Claim 1 stays on D0.

**What the inlet does to the plant (the finding to carry forward).** With the inlet varying uniformly over ±2 K, the plant is at the jacket-capacity limit (Qc ≥ 395 L/min) for a large share of the time, where at the nominal inlet (D0) only Sc5 is: fraction of samples (windows touched) Sc1 **0.31 (0.38)**, Sc2 **0.43 (0.46)**, Sc3 0.12 (0.18), Sc4 0.30 (0.34), Sc5 0.73 (0.76; 0.99 at the nominal inlet), Sc7 0.38 (0.42), against 0.00 in D0 for Sc1–Sc4 and Sc7. The mean coolant flow of the healthy plant is 175 L/min in D1 against 51 L/min in D0. This is the cooling-margin effect of notebook 34 acting over the whole ±2 K range: a hot feed or a warm coolant supply pushes even the healthy plant to the valve limit, where the controller can no longer hold T (mean T − Tsp +0.19 K for Sc1, +0.35 K for Sc2; Sc7 +2.31 K includes its +2 K sensor drift). **Consequences:** every per-scenario number computed on D1 is an average over unsaturated and saturated windows, and the production posterior must read the fault through both regimes (it was trained on exactly this inlet distribution, notebook 36, so the comparison is fair, but the scenario labels hide a heterogeneous mixture); results should be reported with the saturated fraction next to them. The ±2 K range is an assumption about plant operation (daily and seasonal supply variation) and not a measured quantity.

**Use (storyline).** D1 is the dataset of the production posterior (nb36); the headline precision, calibration and diagnosis numbers of Phase 3 come from it. The **naive** posterior on D1 shows the inlet silent failure of notebook 34 on the paper's own data and goes in the sensitivity section only (user decision). D0 stays the nominal-inlet dataset (naive model, Claim 1). *Limitations:* the measurement offsets are drawn per window (as in the nb36 training), whereas a real sensor keeps its calibration offset for months (notebook 10's stream uses persistent offsets, user decision); Ci is unmeasured; the inlet is constant within a window.
